# 🤖 Mini Project — AI Chatbot

## Day 5 | Generative AI & LLM APIs

Build a fully functional AI chatbot with:
- Conversation memory
- Customizable personality via system prompt
- Token usage tracking
- Graceful error handling

### Learning Objectives
- Use LLM APIs programmatically
- Implement conversation memory
- Design effective system prompts
- Handle API errors gracefully

## Step 1: Setup & Installation

In [ ]:
# Install required packages
# !pip install groq

## Step 2: Import Libraries & Configure API

In [ ]:
"""
Import required libraries.

- groq: The Groq SDK for fast LLM inference
- datetime: For timestamping messages
- json: For saving/loading conversation history
"""

from groq import Groq
from datetime import datetime
import json
import os

# Initialize the Groq client
# Get your free API key from: https://console.groq.com
API_KEY = "gsk_your_key_here"  # Replace with your key
client = Groq(api_key=API_KEY)

print("✅ Groq client initialized successfully!")

## Step 3: Design the System Prompt

The system prompt is the **most important part** of any AI application. It defines:
- Who the AI is
- How it should behave
- What rules it should follow
- What it should and shouldn't do

In [ ]:
"""
System Prompt Design — The AI's Personality & Rules

A great system prompt should:
1. Define the role/persona clearly
2. Set behavioral guidelines
3. Include response format preferences
4. Set boundaries (what NOT to do)
"""

SYSTEM_PROMPT = """You are CodeBuddy, a friendly and encouraging AI coding assistant.

PERSONALITY:
- You're patient, friendly, and enthusiastic about coding
- You use simple language and relatable analogies
- You occasionally use emojis to be approachable (but not excessively)
- You celebrate small wins and encourage the learner

RESPONSE STYLE:
- Give concise answers (not essays)
- Always include code examples when relevant
- Use bullet points for clarity
- Explain WHY, not just WHAT
- If showing code, add comments explaining each line

RULES:
- If you don't know something, say so honestly
- For coding questions, always provide working code examples
- If the user asks a non-coding question, gently redirect to coding topics
- Never generate harmful, offensive, or misleading content
- If the user seems frustrated, be extra encouraging
"""

print("✅ System prompt configured!")
print(f"📊 System prompt length: {len(SYSTEM_PROMPT.split())} words")

## Step 4: Build the Chatbot Class

In [ ]:
"""
The AIChatbot class — encapsulates all chatbot functionality.

Key Design Decisions:
- conversation_history: List of all messages (how the bot 'remembers')
- token_usage: Tracks total tokens for cost monitoring
- max_retries: Handles transient API failures gracefully
"""

class AIChatbot:
    """A production-style AI chatbot with memory and error handling."""
    
    def __init__(self, system_prompt: str, model: str = "llama-3.3-70b-versatile"):
        """
        Initialize the chatbot.
        
        Args:
            system_prompt: The AI's personality and rules
            model: Which LLM model to use
        """
        self.model = model
        self.system_prompt = system_prompt
        self.conversation_history = [
            {"role": "system", "content": system_prompt}
        ]
        self.total_tokens_used = 0
        self.message_count = 0
        self.session_start = datetime.now()
    
    def send_message(self, user_message: str, temperature: float = 0.7) -> str:
        """
        Send a message to the AI and get a response.
        
        Args:
            user_message: The user's input text
            temperature: Creativity level (0.0 to 1.0)
        
        Returns:
            The AI's response text
        """
        # Add user message to conversation history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        try:
            # Call the LLM API
            response = client.chat.completions.create(
                model=self.model,
                messages=self.conversation_history,
                temperature=temperature,
                max_tokens=1024
            )
            
            # Extract the AI's response
            ai_message = response.choices[0].message.content
            
            # Save to conversation history (this is the 'memory')
            self.conversation_history.append({
                "role": "assistant",
                "content": ai_message
            })
            
            # Track token usage
            self.total_tokens_used += response.usage.total_tokens
            self.message_count += 1
            
            return ai_message
            
        except Exception as e:
            error_msg = f"Sorry, I encountered an error: {str(e)}"
            # Remove the failed user message from history
            self.conversation_history.pop()
            return error_msg
    
    def get_stats(self) -> str:
        """Get session statistics."""
        duration = (datetime.now() - self.session_start).seconds
        minutes = duration // 60
        seconds = duration % 60
        
        return (
            f"📊 Session Statistics:\n"
            f"   Messages exchanged: {self.message_count}\n"
            f"   Total tokens used: {self.total_tokens_used:,}\n"
            f"   Session duration: {minutes}m {seconds}s\n"
            f"   Model: {self.model}\n"
            f"   History length: {len(self.conversation_history)} messages"
        )
    
    def save_conversation(self, filename: str = "chat_history.json"):
        """Save conversation history to a JSON file."""
        data = {
            "session_start": self.session_start.isoformat(),
            "model": self.model,
            "total_tokens": self.total_tokens_used,
            "messages": self.conversation_history
        }
        with open(filename, "w") as f:
            json.dump(data, f, indent=2)
        return f"💾 Conversation saved to {filename}"
    
    def clear_history(self):
        """Reset conversation (keep system prompt)."""
        self.conversation_history = [
            {"role": "system", "content": self.system_prompt}
        ]
        self.message_count = 0
        self.total_tokens_used = 0
        self.session_start = datetime.now()
        return "🔄 Conversation cleared. Starting fresh!"

print("✅ AIChatbot class defined!")

## Step 5: Test the Chatbot

In [ ]:
# Create chatbot instance
chatbot = AIChatbot(system_prompt=SYSTEM_PROMPT)

# Test conversation
print("=" * 50)
print("🤖 Testing CodeBuddy AI Chatbot")
print("=" * 50)

# Message 1
response1 = chatbot.send_message("Hi! I'm new to Python. Where should I start?")
print(f"\n👤 User: Hi! I'm new to Python. Where should I start?")
print(f"🤖 CodeBuddy: {response1}")

# Message 2 — Tests memory (refers to previous message)
response2 = chatbot.send_message("You mentioned variables. Can you explain what they are?")
print(f"\n👤 User: You mentioned variables. Can you explain what they are?")
print(f"🤖 CodeBuddy: {response2}")

# Message 3 — Tests follow-up
response3 = chatbot.send_message("Show me a simple example with numbers")
print(f"\n👤 User: Show me a simple example with numbers")
print(f"🤖 CodeBuddy: {response3}")

# Show stats
print(f"\n{chatbot.get_stats()}")

## Step 6: Interactive Chat Loop

In [ ]:
"""
Interactive Chat Loop — Run this cell to start chatting!

Commands:
  'quit' or 'exit' — End the session
  'stats'          — View session statistics
  'save'           — Save conversation to file
  'clear'          — Clear conversation history
  'help'           — Show available commands
"""

# Create a fresh chatbot
bot = AIChatbot(system_prompt=SYSTEM_PROMPT)

print("🤖 CodeBuddy — AI Coding Assistant")
print("=" * 45)
print("Type your coding questions! Type 'help' for commands.")
print("=" * 45)

while True:
    user_input = input("\n👤 You: ").strip()
    
    if not user_input:
        continue
    
    # Handle commands
    if user_input.lower() in ['quit', 'exit', 'bye']:
        print(f"\n{bot.get_stats()}")
        print("\n🤖 CodeBuddy: Goodbye! Keep coding! 🚀")
        break
    
    if user_input.lower() == 'stats':
        print(f"\n{bot.get_stats()}")
        continue
    
    if user_input.lower() == 'save':
        print(bot.save_conversation())
        continue
    
    if user_input.lower() == 'clear':
        print(bot.clear_history())
        continue
    
    if user_input.lower() == 'help':
        print("\n📋 Commands: 'quit', 'stats', 'save', 'clear', 'help'")
        continue
    
    # Get AI response
    response = bot.send_message(user_input)
    print(f"\n🤖 CodeBuddy: {response}")

## 🎯 Mini Exercise: Customize Your Chatbot

Try these modifications:

1. **Change the personality** — Make it a pirate, a fitness coach, or a recipe bot
2. **Add a word count limit** — Make responses shorter or longer
3. **Try different temperatures** — Use 0.0 for a factual bot, 1.0 for a creative bot
4. **Add a language preference** — Make it respond in Hindi or Spanish

In [ ]:
# 🏴‍☠️ Exercise: Create a Pirate Coding Tutor!

PIRATE_PROMPT = """
You are Captain CodeBeard, a pirate who teaches Python programming.

RULES:
- Speak like a pirate (arr, matey, ahoy, etc.)
- Use sailing analogies to explain coding concepts
- Still give accurate, helpful coding advice
- Call variables 'treasure chests' and functions 'ship commands'
- End responses with pirate wisdom
"""

pirate_bot = AIChatbot(system_prompt=PIRATE_PROMPT)
response = pirate_bot.send_message("What is a for loop?")
print(f"🏴‍☠️ Captain CodeBeard: {response}")

## 📝 Summary

### What We Built
- ✅ A fully functional AI chatbot with memory
- ✅ Customizable personality via system prompts
- ✅ Token usage tracking
- ✅ Error handling
- ✅ Conversation save/load

### Key Concepts
1. **System Prompt** — Defines AI personality and rules
2. **Conversation History** — List of messages sent with every API call
3. **Token Tracking** — Essential for cost management
4. **Error Handling** — Production apps must handle API failures

### What's Next?
In the **Industry Project**, we'll build an AI Career Mentor — a more sophisticated version with:
- Professional career guidance
- Structured conversation flows
- Session management
- Export features